# Introduction to Large Language Models (LLMs)

Large Language Models (LLMs) have revolutionized natural language processing (NLP) and have become foundational tools in various AI applications. These models, trained on massive datasets, are capable of generating human-like text, translating languages, summarizing documents, answering questions, and much more. In this notebook, we will explore how to use large language models, particularly those built on architectures like GPT, BERT, and LLaMA.

## What Are Large Language Models?

LLMs are a type of deep learning model specifically designed to understand and generate text. They are typically based on transformer architectures, which allow them to process and generate text in parallel, making them highly efficient and powerful. The "large" in LLMs refers to the vast number of parameters—often in the billions—these models possess, which enables them to capture the nuances and complexities of human language.

### Key Features of LLMs:

- **Contextual Understanding:** LLMs can generate text that is contextually relevant, meaning they understand the context in which words are used, leading to more accurate and coherent outputs.
- **Transfer Learning:** These models can be fine-tuned on specific tasks with smaller datasets, making them highly versatile for various NLP applications.
- **Generative Capabilities:** Beyond understanding text, LLMs can generate creative and complex content, from poetry to technical explanations.

## Why Use LLMs?

The widespread adoption of LLMs is driven by their ability to perform a wide range of tasks with minimal human intervention. They are used in chatbots, content creation, sentiment analysis, language translation, code generation, and more. Their ability to generalize across tasks without needing task-specific models has made them invaluable in both industry and research.

## What Will You Learn?

In this notebook, you will learn how to leverage open-source LLMs for text generation and other NLP tasks using Python and popular libraries such as `transformers`. We will guide you through:
- Setting up the environment for working with LLMs.
- Loading pre-trained models and tokenizers.
- Generating text based on specific prompts.

By the end of this notebook, you will have a solid understanding of how to implement and use LLMs in practical scenarios, opening the door to countless AI-powered applications.

Let’s get started!


In [ ]:
!pip install \
  transformers==4.57.3 \
  bitsandbytes==0.49.0 \
  accelerate==1.12.0


# LLM: GaMS3-12B

- [Model details](https://huggingface.co/cjvt/GaMS3-12B-Instruct)
- Completely open, you do not have to apply for access
- We will use a quantized version of the model (preserves accuracy of the model while significantly reduces memory requirements). If you are interested, read more [here](https://huggingface.co/docs/optimum/concept_guides/quantization).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

device = "cuda" # the device to load the model onto
model_id = "cjvt/GaMS3-12B-Instruct"

# Initialize the model with quantization\n",
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Set to True for 4-bit quantization or False for 8-bit
    bnb_4bit_use_double_quant=True,  # Optional: Improves stability in 4-bit quantization
    bnb_4bit_quant_type="nf4",  # Optional: Use 'nf4' for better accuracy or 'fp4' for faster computation
    bnb_4bit_compute_dtype=torch.float16  # Optional: use float16 for better performance on newer GPUs
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map=device,
    quantization_config=bnb_config  # Pass the quantization configuration
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

### GaMS3 Response Generation Function

This function is a small wrapper around a GaMS3 language model that takes a text prompt and returns the model’s answer. From the user’s perspective, it behaves like a simple “ask a question, get an answer” interface. Internally, however, several steps are required to transform natural language into a form the model can process and then convert the model’s output back into readable text.

The model itself does not understand text directly. Instead, all input text is first converted into numbers using a **tokenizer**, which breaks the text into smaller pieces (tokens) such as words or subwords and maps them to numerical IDs. These numbers are what the neural network actually operates on.

Once the prompt is tokenized, it is passed through the GaMS3 transformer model, which has been trained to predict the most likely next token given everything it has seen so far. Generation happens step by step: the model predicts the next token, appends it to the sequence, and repeats this process until a complete response is produced.

Finally, the generated token IDs are converted back into human-readable text using the tokenizer in reverse, and only the newly generated part of the sequence is returned as the final answer.


**Workflow overview:**

1. **Construct chat messages**  
   A system message defines the assistant’s behavior (GaMS does not support system messages, so the system prompt has to be added to the user message), and a user message contains the input prompt.

2. **Apply the chat template**  
   The tokenizer formats the messages into the model-specific chat prompt and appends a generation marker.

3. **Tokenize and move to device**  
   The formatted prompt is converted into input IDs and transferred to the target device (CPU or GPU).

4. **Generate model output**  
   The model produces up to a fixed number of new tokens as a continuation of the prompt.

5. **Extract generated content only**  
   The original prompt tokens are removed so that only newly generated tokens remain.

6. **Decode to text**  
   Token IDs are converted back into readable text, skipping special tokens.

The function returns the final text response generated by the GaMS3 model.


In [ ]:
# A function that returns the answer from the model
def gams3_generate(prompt):
    messages = [
      {"role": "user", "content": prompt}
  ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

In [ ]:
# Chat about anything
prompt = "V 3 stavkih mi na kratko predstavi veliki jezikovni model."
response = gams3_generate(prompt)
print(response)

### Modify text generation parameters

Now, we are going to change the hyperparameters of text generation. We will focus on only a few most important. You can find a full list of parameters on [HuggingFace](https://huggingface.co/docs/transformers/main_classes/text_generation).

In [ ]:
prompt = "Pripravi osnutek objave za LinkedIn, v kateri podjetje napoveduje novo sodelovanje na področju umetne inteligence."
messages = [
  {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(device)

### Temperature
**Temperature** is a parameter used in natural language processing models to increase or decrease the “confidence” a model has in its most likely response.

A low temperature makes the model behave more conservatively: it strongly prefers the most likely words, which leads to predictable, focused, and often more factual responses.

A higher temperature loosens this preference, allowing less likely words to be chosen more often, which can make the output more creative, varied, or surprising—but also more prone to errors or incoherence.

You can think of temperature like adjusting the “randomness dial”: turning it down produces safe, consistent answers, while turning it up encourages exploration and diversity in what the model says.

You can find a great visualization and explanation of temperature [here](https://lukesalamone.github.io/posts/what-is-temperature/).


In [ ]:
# change temperature of the model
for tmp in [0.1, 0.5, 0.98]:  # lower values make model more deterministic
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=512,
        do_sample=True,
        temperature=tmp
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    print("Temperature:", tmp)
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(response)

### Top-k Sampling

**Top-k sampling** controls *which words the model is even allowed to choose from* when generating the next token.

At every step, the model assigns probabilities to many possible next words. Instead of considering *all* of them, top-k sampling **keeps only the k most likely candidates** and completely ignores the rest. The next word is then sampled only from this reduced shortlist.

When the context is **short or vague** (e.g., after “The”), probability is spread across many plausible words (“dog,” “car,” “woman,” etc.). In this case, the top-k set may capture only part of the total probability mass. As the context becomes **more specific** (e.g., “The car”), probability concentrates on a few highly likely words (“drives,” “is”), and the top-k candidates cover almost all reasonable continuations.

A **small k** makes the model more conservative and predictable, because it can choose only from a few high-probability words. A **larger k** allows more variety, enabling less common but still plausible words to appear, which increases diversity but also the risk of awkward or less accurate output.

In practice, top-k sampling acts as a **hard filter on creativity**: it limits how far the model can stray by restricting the choice set, while temperature (often used together with top-k) controls how strongly the model prefers the top options *within* that set.


![top_k_sampling](https://raw.githubusercontent.com/patrickvonplaten/scientific_images/master/top_k_sampling.png)

In [ ]:
# change top_k of the model
for v in [10, 50, 100]:
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=512,
        do_sample=True,
        top_k=v
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    print("Top K:", v)
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(response)